In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.experimental import enable_iterative_imputer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, validation_curve, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, IterativeImputer, SimpleImputer
from sklearn.feature_selection import SelectFromModel,SequentialFeatureSelector

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import StackingClassifier

from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix,roc_curve, auc



# Exploratory Data Analysis

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

In [ ]:
print(train_df.columns.values)

In [ ]:
train_df.head()

In [ ]:
train_df.tail()

In [ ]:
train_df.info()

In [ ]:
tmp_df = pd.crosstab(index=train_df['Pclass'], columns = train_df['Survived'], normalize = 'index')
ax = tmp_df.plot(kind='bar',stacked=True,color=['#ff9999', '#66b3ff'],rot=0)
for container in ax.containers:
    ax.bar_label(container, padding=3, label_type='center', fmt='%.2f')

plt.title('Survival Distribution by Passenger Class',fontweight='bold',pad=15)
plt.legend(['Died (0)', "Survived(1)"],loc='upper right', bbox_to_anchor=(1.38,1))
plt.xlabel('Passenger Class (Pclass)', fontsize=10)
plt.ylabel('Percentage of Passengers', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
tmp_df = pd.crosstab(index= train_df['Sex'], columns= train_df['Survived'], normalize=True)
ax = tmp_df.plot(kind='bar', stacked=True,color=['#ff9999', '#66b3ff'])
for container in ax.containers:
    ax.bar_label(container, padding=3, label_type='center',fmt='%.2f')

ax.tick_params(axis='x', rotation=0)
plt.xlabel('Gender', fontsize=10)
plt.ylabel('Percentage of Gender', fontsize=10)
plt.legend(['Died(0)', 'Survived(1)'],loc='best')
plt.title('Survival Distribution by Gender',pad=15,fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
tmp_df = train_df[["SibSp", "Survived"]].groupby('SibSp', as_index=False).mean().round(2).sort_values(by='Survived', ascending=False).reset_index(drop=True)
display(tmp_df)

In [ ]:
tmp_df = train_df[['Parch', 'Survived']].groupby('Parch', as_index=False).mean().round(2).sort_values(by='Survived', ascending=False).reset_index(drop=True)
display(tmp_df)

In [ ]:
g = sns.FacetGrid(train_df, col='Survived')
g.map(plt.hist,'Age',bins=20)
g.figure.suptitle('Survival Distribution by Age',fontweight = 'bold', y=1.1)
plt.show()

In [ ]:
g = sns.FacetGrid(train_df,row = 'Survived', col = 'Pclass')
g.map(sns.countplot, 'Sex',order=['female', 'male'])
plt.show()

In [ ]:
g = sns.FacetGrid(train_df, col='Survived')
g.map(plt.hist,'Fare',bins=5)
g.figure.suptitle('Survival Distribution by Fare',fontweight = 'bold', y=1.1)
plt.show()

## Observation:
- Few elderly passengers of age 65+
- Infants, Female and Old passengers were prioritized to save
- Pclass1 had the highest survival rate.
- Those who had Parent / Children of 3 had better survival rate comparatively

# Data Preprocessing

In [ ]:
train_df['Deck'] = train_df['Cabin'].str[0]
train_df['Deck'] = train_df['Deck'].fillna('Unknown')
train_df['Deck'].value_counts()

In [ ]:
train_df['Deck'] = train_df['Cabin'].str[0]
train_df['Deck'] = train_df['Deck'].fillna('Unknown')
rare_decks = ['A', 'G', 'F', 'T']
train_df['Deck'] = train_df['Deck'].apply(lambda x : 'Rare' if x in rare_decks else x) 


train_df['Fam_size'] = train_df['Parch'] + train_df['SibSp']

train_df['IsAlone'] = 0
train_df.loc[train_df['Fam_size'] == 0,'IsAlone' ] = 1

In [ ]:
train_df.isnull().sum()

In [ ]:
X = train_df.drop(columns=['PassengerId', 'Survived', 'Cabin', 'Ticket', 'Name', 'SibSp', 'Parch'])
y = train_df['Survived']

X_train, X_cv, y_train, y_cv = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)

print(f'X_train shape:',X_train.shape)
print(f'X_cv shape:',X_cv.shape,'\n')

print(f'y_train shape:',y_train.shape)
print(f'y_cv shape:',y_cv.shape,'\n')

print(f'y_train class distribution',np.bincount(y_train.values))
print(f'y_cv class distribution',np.bincount(y_cv.values))

In [ ]:
categorical_features = X_train.select_dtypes(include='object').columns.tolist()
numerical_features = X_train.select_dtypes(include='number').columns.tolist()

numerical_transformer = Pipeline(steps = [
    ('imputer', IterativeImputer(max_iter=10,random_state=42)),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('OHE', OneHotEncoder(handle_unknown='ignore',sparse_output=False,drop='first'))])

binary_transformer = Pipeline(steps = [
    ('OE', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value=-1))])

preprocessor = ColumnTransformer(transformers = [
    ('numerical_transformer', numerical_transformer, ['Pclass', 'Age', 'Fam_size' , 'Fare']),
    ('categorical_transformer', categorical_transformer, ['Embarked', 'Deck']),
    ('binary_transformer', binary_transformer, ['Sex']),
    ],remainder = 'passthrough')

# Feature Selection

In [ ]:
selector_logreg = LogisticRegression(penalty='l1', max_iter=500, solver='liblinear', class_weight='balanced')
param_range = [0.001,0.01,0.1,1,10,100]
pipe_logreg = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('feature_selector', selector_logreg)
])
train_scores, test_scores = validation_curve(estimator=pipe_logreg,
                                             X = X_train,
                                             y = y_train,
                                             param_name = 'feature_selector__C',
                                             param_range = param_range,
                                             cv = 5,
                                             scoring = 'roc_auc')
train_mean = np.mean(train_scores,axis=1)
train_std = np.std(train_scores,axis=1)
test_mean = np.mean(test_scores,axis=1)
test_std = np.std(test_scores,axis=1)

plt.plot(param_range, train_mean, color='blue', marker='o', markersize=5, label='Training score')
plt.plot(param_range, test_mean, color='green',marker='s', linestyle='--',label='Validation score')
plt.fill_between(param_range, train_mean + train_std, train_mean - train_std, alpha=0.15,color='blue')
plt.fill_between(param_range, test_mean + test_std, test_mean - test_std, alpha=0.15, color='green')

plt.grid()
plt.xscale('log')
plt.legend(loc='lower right')
plt.xlabel('Parameters of C')
plt.ylabel('ROC-AUC score')
plt.ylim([0.5,0.9])
plt.tight_layout()
plt.show()

In [ ]:
pipe_logreg.set_params(feature_selector__C = 0.1)
pipe_logreg.fit(X_train, y_train)

feature_names = pipe_logreg.named_steps['preprocessing'].get_feature_names_out()
coefficients = pipe_logreg.named_steps['feature_selector'].coef_[0]

feature_importance = pd.DataFrame({
    'Features' : feature_names,
    'Coefficient' : coefficients
})

selected_feature = feature_importance[feature_importance['Coefficient'] != 0.0].reset_index(drop=True)
selected_feature

In [ ]:
rf = RandomForestClassifier(n_estimators=500,random_state=42,class_weight='balanced')
selector_rf = SelectFromModel(rf, threshold=0.06)
pipe_rf = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('feature_selector', selector_rf)
])

pipe_rf.fit(X_train, y_train)

feature_names = pipe_rf.named_steps['preprocessing'].get_feature_names_out()
feature_mask = pipe_rf.named_steps['feature_selector'].get_support()
rf_importance = pipe_rf.named_steps['feature_selector'].estimator_.feature_importances_

rf_feature_importance = pd.DataFrame({
    'Features': feature_names,
    'Importance': rf_importance,
    'Selected': feature_mask
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)


rf_feature_importance[rf_feature_importance['Selected'] == True]

# Model Training

In [ ]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_cv_preprocessed = preprocessor.transform(X_cv)

dt = DecisionTreeClassifier(criterion='entropy',max_depth = 2, random_state=42, min_samples_split = 8, min_samples_leaf = 4)

base_models = {
    'KNN' : KNeighborsClassifier(n_jobs=-1,metric='minkowski',p=2),
    'Logistic Regression' : LogisticRegression(n_jobs=-1, class_weight='balanced', max_iter=3000, random_state=42, penalty = 'l2'),
    'Random Forest' : RandomForestClassifier(n_jobs=-1, random_state=42),
    'SVM' : SVC(probability=True, random_state=42),
    'AdaBoost' : AdaBoostClassifier(estimator = dt, learning_rate=0.01, random_state=42)
}

param_distributions = {
    'KNN' : {"classifier__n_neighbors" : [3, 4, 5, 6], 'classifier__weights' : ['uniform', 'distance']},
    'Logistic Regression' : {"classifier__C" : [0.001, 0.01, 0.1, 1, 10], "classifier__solver" : ['saga', 'lbfgs', 'newton-cholesky']},
    'Random Forest' : {"classifier__n_estimators" : [100, 200, 300], "classifier__max_depth" : [2, 3, 5], "classifier__min_samples_split" : [3, 5, 9], "classifier__min_samples_leaf" : [2, 4, 6]},
    'SVM' : {'classifier__C' : [0.01, 0.1, 1, 10], 'classifier__gamma' : ['scale', 'auto'], 'classifier__kernel' : ['rbf']},
    'AdaBoost' : {"classifier__n_estimators" : [50, 100, 200]}
}

tuned_models = []
for name, model in base_models.items():
    print(f'--- Tuning & Evaluating: {name} ---')
    selector_rf = SelectFromModel(RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced'), threshold=0.06)
    fold_pipeline = Pipeline(steps=[
        ('feature_selector', selector_rf),
        ('classifier', model)
    ])
    gs = GridSearchCV(estimator = fold_pipeline, scoring='roc_auc', cv = 2, n_jobs=-1, param_grid=param_distributions[name])
    scores = cross_val_score(gs, X_train_preprocessed, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    print(f'CV score: {np.mean(scores):.2f} +/- {np.std(scores):.2f}')
    gs.fit(X_train_preprocessed, y_train)
    print(f'Best params for {name}: {gs.best_params_}\n')
    tuned_models.append((name, gs.best_estimator_))

In [ ]:
X_train_selected = selector_rf.fit_transform(X_train_preprocessed, y_train)
X_cv_selected = selector_rf.transform(X_cv_preprocessed)

X_train_tuning, X_eval_tuning, y_train_tuning, y_eval_tuning = train_test_split(
    X_train_selected, y_train, test_size=0.10, random_state=42, stratify=y_train
)

param_distributions_xgb = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6, 7],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9], 
    'reg_alpha': [0, 0.1, 1, 5, 10],          
    'reg_lambda': [0.1, 1, 5, 10, 50]        
}

xgb_base = XGBClassifier(
    n_estimators=1000,           
    early_stopping_rounds=25,    
    eval_metric='logloss',     
    random_state=42,
    n_jobs=-1
)

print("--- Optimizing Advanced XGBoost Hyperparameters ---")

rs_xgb = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions_xgb,
    n_iter=100,
    scoring='roc_auc',
    random_state=42,
    cv=3,
    n_jobs=-1)

rs_xgb.fit(
    X_train_tuning, y_train_tuning,
    eval_set=[(X_eval_tuning, y_eval_tuning)],
    verbose=False 
)

print(f"Optimal Advanced Params: {rs_xgb.best_params_}")
print(f"Best Score: {rs_xgb.best_score_:.2f}")

In [ ]:
XGBoost = XGBClassifier(
    n_estimators=120, 
    learning_rate=0.2,
    max_depth=5,
    min_child_weight=1,
    subsample=0.7,
    colsample_bytree=0.6,
    reg_alpha=0,
    reg_lambda=10,
    random_state=42,
    n_jobs=-1
)

XGB_pipeline = Pipeline(steps=[
    ('feature_selector', selector_rf),
    ('classifier',XGBoost)
])

XGB_pipeline.fit(X_train_preprocessed,y_train)

tuned_models.append(('XGBoost', XGB_pipeline))
print("XGBoost pipeline successfully integrated into tuned_models!")

In [ ]:
joblib.dump(tuned_models, 'tuned_titanic_models.joblib',compress=3)
print("Successfully saved all 6 tuned model pipelines!")

In [ ]:
models = joblib.load('/kaggle/working/tuned_titanic_models.joblib')
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 11))
axes = axes.flatten()

for idx, (name, model) in enumerate(models):
    ax = axes[idx]
    y_train_probs = model.predict_proba(X_train_preprocessed)[:,1]
    fpr_train, tpr_train, thresholds = roc_curve(y_train,y_train_probs)
    auc_train = auc(fpr_train, tpr_train)
    ax.plot(fpr_train, tpr_train, linestyle='--', linewidth=2, label=f"Train (AUC = {auc_train:.2f})", color='c')
    y_cv_probs = model.predict_proba(X_cv_preprocessed)[:,1]
    fpr_cv, tpr_cv, thresholds = roc_curve(y_cv,y_cv_probs)
    auc_cv = auc(fpr_cv, tpr_cv)
    ax.plot(fpr_cv, tpr_cv, linestyle='-', linewidth=2, label=f"Validation (AUC = {auc_cv:.2f})",color='k')
    ax.plot([0, 1], [0, 1], color='gray', linestyle=':', linewidth=1) # Diagonal random guess line
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    if idx >= 3:
        ax.set_xlabel('False Positive Rate', fontsize=10)
    if idx in [0, 3]:
        ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('Individual Model Comparison\nTraining ROC (Cyan) vs. Holdout Validation ROC (Black)', 
             fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Instantiate the regularized models 
RandomForest_regularized = RandomForestClassifier(n_estimators=200, max_depth=4,
    min_samples_split=10, min_samples_leaf=5,
    max_features='sqrt', bootstrap=True,
    max_samples=0.7, random_state=42, n_jobs=-1)

XGBoost_regularized = XGBClassifier(n_estimators=150, learning_rate=0.05,
    max_depth=3, min_child_weight=5,
    subsample=0.6, colsample_bytree=0.6, reg_alpha=1,
    reg_lambda=15, random_state=42, n_jobs=-1)

# 2. Package them into pipelines using your existing selector_rf step
RF_pipeline_new = Pipeline(steps=[
    ('feature_selector', selector_rf),
    ('classifier', RandomForest_regularized)
])

XGB_pipeline_new = Pipeline(steps=[
    ('feature_selector', selector_rf),
    ('classifier', XGBoost_regularized)
])

# 3. Fit the new pipelines on 100% of the preprocessed training data
print("--- Fitting Regularized Models ---")
RF_pipeline_new.fit(X_train_preprocessed, y_train)
XGB_pipeline_new.fit(X_train_preprocessed, y_train)

# 4. Clean out the old 'Random Forest' and 'XGBoost' entries from tuned_models
tuned_models = [model for model in tuned_models if model[0] not in ['Random Forest', 'XGBoost']]

# 5. Append the fresh, regularized pipelines using the exact same original string identifiers
tuned_models.append(('Random Forest', RF_pipeline_new))
tuned_models.append(('XGBoost', XGB_pipeline_new))
print("Successfully replaced old models with regularized versions in tuned_models!")

# 6. Overwrite saved joblib file
joblib.dump(tuned_models, 'tuned_titanic_models.joblib', compress=3)
print(f"Successfully updated and saved all {len(tuned_models)} model pipelines!")

In [ ]:
models = joblib.load('/kaggle/working/tuned_titanic_models.joblib')
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 11))
axes = axes.flatten()

for idx, (name, model) in enumerate(models):
    ax = axes[idx]
    y_train_probs = model.predict_proba(X_train_preprocessed)[:,1]
    fpr_train, tpr_train, thresholds = roc_curve(y_train,y_train_probs)
    auc_train = auc(fpr_train, tpr_train)
    ax.plot(fpr_train, tpr_train, linestyle='--', linewidth=2, label=f"Train (AUC = {auc_train:.2f})", color='c')
    y_cv_probs = model.predict_proba(X_cv_preprocessed)[:,1]
    fpr_cv, tpr_cv, thresholds = roc_curve(y_cv,y_cv_probs)
    auc_cv = auc(fpr_cv, tpr_cv)
    ax.plot(fpr_cv, tpr_cv, linestyle='-', linewidth=2, label=f"Validation (AUC = {auc_cv:.2f})",color='k')
    ax.plot([0, 1], [0, 1], color='gray', linestyle=':', linewidth=1) # Diagonal random guess line
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    if idx >= 3:
        ax.set_xlabel('False Positive Rate', fontsize=10)
    if idx in [0, 3]:
        ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('Individual Model Comparison\nTraining ROC (Cyan) vs. Holdout Validation ROC (Black)', 
             fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

In [ ]:
models = joblib.load('/kaggle/working/tuned_titanic_models.joblib')
voting_clf = VotingClassifier(estimators = models, n_jobs=-1, voting='soft')
print("--- Calculating Voting Classifier Cross-Validation Score ---")

voting_scores = cross_val_score(estimator=voting_clf, X=X_train_preprocessed, y=y_train,cv=5, scoring='roc_auc')
print(f"Voting Classifier AUC Scores: {voting_scores.mean():.2f} +/-{voting_scores.std():.2f} ")
voting_clf.fit(X_train_preprocessed, y_train)

In [ ]:
for name, pipeline in models:
    scores = cross_val_score(pipeline, X_train_preprocessed, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    print(f"{name} Individual AUC: {scores.mean():.4f}")

In [ ]:
good_model_names = ['KNN', 'Logistic Regression', 'XGBoost']
good_models = [(name, pipeline) for name, pipeline in models if name in good_model_names]
voting_clf_good = VotingClassifier(estimators = good_models, n_jobs=-1, voting='soft')

voting_clf_good_scores = cross_val_score(estimator=voting_clf_good, X=X_train_preprocessed, y=y_train,cv=5, scoring='roc_auc')
print(f"Voting Classifier AUC Scores: {voting_clf_good_scores.mean():.2f} +/-{voting_clf_good_scores.std():.2f} ")
voting_clf_good.fit(X_train_preprocessed, y_train)

In [ ]:
models = joblib.load('/kaggle/working/tuned_titanic_models.joblib')
good_model_names = ['KNN', 'Logistic Regression', 'XGBoost']
good_models = [(name, pipeline) for name, pipeline in models if name in good_model_names]

meta_learner = LogisticRegression(C=0.1, random_state=42)

stacking_clf = StackingClassifier(estimators=good_models,final_estimator = meta_learner, cv=5, n_jobs=-1)
stacking_scores = cross_val_score(
    estimator=stacking_clf, 
    X=X_train_preprocessed, 
    y=y_train, 
    cv=5, 
    scoring='roc_auc', 
    n_jobs=-1
)

print(f"Stacking AUC Score: {stacking_scores.mean():.4f} +/- {stacking_scores.std():.2f}")
stacking_clf.fit(X_train_preprocessed, y_train)

In [ ]:
ensembles = {
    "Stacking Classifier": stacking_clf,
    "Voting Classifier": voting_clf,
    "Voting Classifier_top_3": voting_clf_good
}

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18,5))


for idx, (name, model) in enumerate(ensembles.items()):
    y_pred = model.predict(X_cv_preprocessed)
    print(f'{name}')
    print(classification_report(y_cv, y_pred,target_names=['Died (0)', 'Survived (1)']))
    print("-"*70)
    cm = confusion_matrix(y_cv, y_pred)
    ax = axes[idx]
    sns.heatmap(cm, annot=True,ax=ax, cbar=False, xticklabels= ['Predicted Died', 'Predicted Survived'], yticklabels=['Actual Died', 'Actual Survived'],fmt='d')
    ax.set_title(f"{name}",fontweight='bold')
    ax.set_xlabel('Predicted Labels',fontweight='bold')

axes[0].set_ylabel('Actual Labels',fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline

test_df = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

test_df['Deck'] = test_df['Cabin'].str[0].fillna('Unknown')
rare_decks = ['A', 'G', 'F', 'T']
test_df['Deck'] = test_df['Deck'].apply(lambda x : 'Rare' if x in rare_decks else x) 

test_df['Fam_size'] = test_df['Parch'] + test_df['SibSp']

test_df['IsAlone'] = 0
test_df.loc[test_df['Fam_size'] == 0, 'IsAlone'] = 1

X_train_full = pd.concat([X_train, X_cv], axis=0)
y_full = pd.concat([y_train, y_cv], axis=0)

submission_pipeline = Pipeline(steps = [
    ('preprocess', preprocessor),
    ('model', voting_clf_good)
])

submission_pipeline.fit(X_train_full, y_full)

pipeline_features = ['Pclass', 'Age', 'Fam_size', 'Fare', 'Embarked', 'Deck', 'Sex', 'IsAlone']
test_df_cleaned = test_df[pipeline_features]

y_test_pred = submission_pipeline.predict(test_df_cleaned)

submission = pd.DataFrame({
    'PassengerId' : test_df['PassengerId'],
    'Survived' : y_test_pred
})

submission.to_csv('submission.csv', index=False)
print("Submission file successfully saved to 'submission.csv'!")